# GP fit for ALICE Pb-Pb 2.76 TeV universality data, Grad

This notebook fits one selected centrality from the `Grad` observables in:

`/Users/avilamrs/Remote-HPC/code-space/my-work/hic-bayes/data/AlicePbPb2760/universality_PbPb2760/Grad`

The selected centrality is `0-5%` by default.

The experimental files provide central values and one uncertainty column only. Here that uncertainty is treated as the experimental uncorrelated/statistical uncertainty.

The GP fit uses only the central values. The experimental uncertainties are kept only for comparison with the GP white-noise term.

Model:

$$\log U(x_T)_i = f(x_{T,i}) + \epsilon_i,$$

$$f \sim \mathcal{GP}(0, \sigma_f^2 R_{\ell=0.2}), \qquad \epsilon_i \sim \mathcal{N}(0, \sigma_n^2).$$

The RBF correlation length is fixed at $\ell=0.2$. The kernel amplitude $\sigma_f$ and white-noise amplitude $\sigma_n$ are fitted by marginal likelihood.

In [2]:
from pathlib import Path
import csv
import uuid

import numpy as np
from IPython.display import HTML, display

try:
    from scipy.optimize import minimize
except ImportError:
    minimize = None

DATA_DIR = Path("/Users/avilamrs/Remote-HPC/code-space/my-work/hic-bayes/data/AlicePbPb2760/universality_PbPb2760/Grad")
EXPERIMENT_FILE = DATA_DIR / "experiment.csv"
EXP_ERROR_FILE = DATA_DIR / "exp_error.csv"
METADATA_FILE = DATA_DIR / "observable_metadata.csv"
SELECTED_CENTRALITY = "0-5%"
LENGTH_SCALE = 0.2
JITTER = 1e-10

In [3]:
def read_one_row_csv(path):
    with path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.reader(f))
    if len(rows) < 2:
        raise ValueError(f"Expected header plus one data row in {path}")
    return rows[0], np.asarray([float(x) for x in rows[1]], dtype=float)


def read_metadata(path):
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        return list(reader)


columns, y_all = read_one_row_csv(EXPERIMENT_FILE)
error_columns, exp_uncorr_all = read_one_row_csv(EXP_ERROR_FILE)
metadata = read_metadata(METADATA_FILE)

if columns != error_columns:
    raise ValueError("experiment.csv and exp_error.csv columns do not match")
if len(columns) != len(metadata):
    raise ValueError("metadata row count does not match experiment columns")

records = []
for idx, (column, y_value, exp_error, meta) in enumerate(zip(columns, y_all, exp_uncorr_all, metadata)):
    if column != meta["column"]:
        raise ValueError(f"Column mismatch at index {idx}: {column} != {meta['column']}")
    records.append({
        "index": idx,
        "column": column,
        "observable": meta["observable"],
        "system": meta["system"],
        "experiment": meta["experiment"],
        "centrality": meta["centrality"],
        "centrality_key": meta["centrality_key"],
        "centrality_index": int(meta["centrality_index"]),
        "bin": int(meta["bin"]),
        "x_T": float(meta["x_T"]),
        "y": y_value,
        "exp_uncorr": exp_error,
    })

centralities = sorted({r["centrality"] for r in records}, key=lambda c: min(r["centrality_index"] for r in records if r["centrality"] == c))
print(f"Loaded {len(records)} Grad points")
print(f"Available centralities: {centralities}")
print(f"Selected centrality: {SELECTED_CENTRALITY}")
if SELECTED_CENTRALITY not in centralities:
    raise ValueError(f"Selected centrality {SELECTED_CENTRALITY!r} not found")
print("The GP fit uses only x_T and central values; exp_error.csv is used only for comparison.")

Loaded 287 Grad points
Available centralities: ['0-5%', '5-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50-60%']
Selected centrality: 0-5%
The GP fit uses only x_T and central values; exp_error.csv is used only for comparison.


In [4]:
def rbf_correlation(x1, x2, length_scale=LENGTH_SCALE):
    x1 = np.asarray(x1, dtype=float).reshape(-1, 1)
    x2 = np.asarray(x2, dtype=float).reshape(1, -1)
    return np.exp(-0.5 * ((x1 - x2) / length_scale) ** 2)


def stable_cholesky(K):
    jitter = JITTER
    for _ in range(8):
        try:
            return np.linalg.cholesky(K + jitter * np.eye(K.shape[0]))
        except np.linalg.LinAlgError:
            jitter *= 10.0
    raise np.linalg.LinAlgError("Cholesky decomposition failed even after adding jitter")


def fit_gp_fixed_ell(x, y, length_scale=LENGTH_SCALE):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    log_y = np.log(y)
    z = log_y - np.mean(log_y)
    R = rbf_correlation(x, x, length_scale=length_scale)

    def nll(theta):
        log_sigma_f, log_sigma_n = theta
        sigma_f = np.exp(log_sigma_f)
        sigma_n = np.exp(log_sigma_n)
        K = sigma_f**2 * R + sigma_n**2 * np.eye(len(x))
        L = stable_cholesky(K)
        alpha = np.linalg.solve(L.T, np.linalg.solve(L, z))
        log_det = 2.0 * np.sum(np.log(np.diag(L)))
        return 0.5 * z @ alpha + 0.5 * log_det + 0.5 * len(x) * np.log(2.0 * np.pi)

    z_std = max(float(np.std(z)), 1e-8)
    initial = np.log([z_std, 0.05 * z_std])

    if minimize is not None:
        result = minimize(
            nll,
            initial,
            method="Nelder-Mead",
            options={"maxiter": 5000, "xatol": 1e-10, "fatol": 1e-10},
        )
        if result.success and np.isfinite(result.fun):
            sigma_f, sigma_n = np.exp(result.x)
            fit_method = "scipy minimize"
            nll_value = float(result.fun)
        else:
            sigma_f = sigma_n = nll_value = None
            fit_method = None
    else:
        sigma_f = sigma_n = nll_value = None
        fit_method = None

    if fit_method is None:
        sigma_f_grid = np.geomspace(0.05 * z_std, 5.0 * z_std, 80)
        sigma_n_grid = np.geomspace(1e-4 * z_std, 1.0 * z_std, 80)
        best = None
        for sigma_f_try in sigma_f_grid:
            for sigma_n_try in sigma_n_grid:
                value = nll(np.log([sigma_f_try, sigma_n_try]))
                if best is None or value < best[0]:
                    best = (value, sigma_f_try, sigma_n_try)
        nll_value, sigma_f, sigma_n = best
        fit_method = "grid search fallback"

    K_corr_log = sigma_f**2 * R
    K_delta_log = sigma_n**2 * np.eye(len(x))
    K_total_log = K_corr_log + K_delta_log
    L = stable_cholesky(K_total_log)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, z))

    K_train_star = sigma_f**2 * R
    log_mean_train = np.mean(log_y) + K_train_star @ alpha
    v_train = np.linalg.solve(L, K_train_star.T)
    log_corr_var_train = np.maximum(sigma_f**2 - np.sum(v_train**2, axis=0), 0.0)

    gp_mean_train = np.exp(log_mean_train)
    gp_corr_std_train = gp_mean_train * np.sqrt(log_corr_var_train)
    gp_uncorr_std_train = gp_mean_train * sigma_n

    x_grid = np.linspace(x.min(), x.max(), 500)
    R_star = rbf_correlation(x_grid, x, length_scale=length_scale)
    K_star = sigma_f**2 * R_star
    log_mean_grid = np.mean(log_y) + K_star @ alpha
    v_grid = np.linalg.solve(L, K_star.T)
    log_corr_var_grid = np.maximum(sigma_f**2 - np.sum(v_grid**2, axis=0), 0.0)
    gp_mean_grid = np.exp(log_mean_grid)
    gp_corr_std_grid = gp_mean_grid * np.sqrt(log_corr_var_grid)
    gp_uncorr_std_grid = gp_mean_grid * sigma_n

    return {
        "x": x,
        "y": y,
        "log_y": log_y,
        "R": R,
        "sigma_f": sigma_f,
        "sigma_n": sigma_n,
        "sigma_f2": sigma_f**2,
        "sigma_n2": sigma_n**2,
        "nll": nll_value,
        "fit_method": fit_method,
        "gp_mean_train": gp_mean_train,
        "gp_corr_std_train": gp_corr_std_train,
        "gp_uncorr_std_train": gp_uncorr_std_train,
        "x_grid": x_grid,
        "gp_mean_grid": gp_mean_grid,
        "gp_corr_std_grid": gp_corr_std_grid,
        "gp_uncorr_std_grid": gp_uncorr_std_grid,
    }


def safe_ratio(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    return np.divide(numerator, denominator, out=np.full_like(numerator, np.nan, dtype=float), where=denominator != 0)

In [5]:
group = sorted([r for r in records if r["centrality"] == SELECTED_CENTRALITY], key=lambda r: r["x_T"])
x = np.asarray([r["x_T"] for r in group], dtype=float)
y = np.asarray([r["y"] for r in group], dtype=float)
exp_uncorr = np.asarray([r["exp_uncorr"] for r in group], dtype=float)

fit = fit_gp_fixed_ell(x, y, length_scale=LENGTH_SCALE)
fit["exp_uncorr"] = exp_uncorr
fit["records"] = group

print(f"Fitted Grad centrality {SELECTED_CENTRALITY} with fixed ell = {LENGTH_SCALE}")
print(f"n={len(x)}")
print(f"sigma_f={fit['sigma_f']:.6g}, sigma_f^2={fit['sigma_f2']:.6g}")
print(f"sigma_n={fit['sigma_n']:.6g}, sigma_n^2={fit['sigma_n2']:.6g}")
print(f"NLL={fit['nll']:.6g}, method={fit['fit_method']}")

Fitted Grad centrality 0-5% with fixed ell = 0.2
n=41
sigma_f=1.25485, sigma_f^2=1.57466
sigma_n=0.00770553, sigma_n^2=5.93751e-05
NLL=12.0971, method=scipy minimize


In [6]:
def hyperparameter_table_html():
    rel_exp_uncorr = safe_ratio(exp_uncorr, y)
    rel_gp_uncorr = safe_ratio(fit["gp_uncorr_std_train"], fit["gp_mean_train"])
    return f"""
    <table>
      <thead>
        <tr>
          <th>centrality</th>
          <th>n points</th>
          <th>ell</th>
          <th>GP sigma_f log(y)</th>
          <th>GP sigma_f^2 log(y)</th>
          <th>GP sigma_n log(y)</th>
          <th>GP sigma_n^2 log(y)</th>
          <th>mean exp stat / y</th>
          <th>mean (exp stat / y)^2</th>
          <th>mean GP uncorr / GP mean</th>
          <th>NLL</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td>{SELECTED_CENTRALITY}</td>
          <td>{len(y)}</td>
          <td>{LENGTH_SCALE:.3g}</td>
          <td>{fit['sigma_f']:.6g}</td>
          <td>{fit['sigma_f2']:.6g}</td>
          <td>{fit['sigma_n']:.6g}</td>
          <td>{fit['sigma_n2']:.6g}</td>
          <td>{np.nanmean(rel_exp_uncorr):.6g}</td>
          <td>{np.nanmean(rel_exp_uncorr**2):.6g}</td>
          <td>{np.nanmean(rel_gp_uncorr):.6g}</td>
          <td>{fit['nll']:.6g}</td>
        </tr>
      </tbody>
    </table>
    """


def point_table_html():
    rows = []
    gp_mean = fit["gp_mean_train"]
    gp_uncorr = fit["gp_uncorr_std_train"]
    for i, r in enumerate(group):
        ratio_mean = safe_ratio(gp_mean[i], r["y"])
        ratio_uncorr = safe_ratio(gp_uncorr[i], exp_uncorr[i])
        rows.append(
            "<tr>"
            f"<td>{i}</td>"
            f"<td>{SELECTED_CENTRALITY}</td>"
            f"<td>{r['bin']}</td>"
            f"<td>{r['x_T']:.6g}</td>"
            f"<td>{r['y']:.6g}</td>"
            f"<td>{exp_uncorr[i]:.6g}</td>"
            f"<td>{gp_mean[i]:.6g}</td>"
            f"<td>{ratio_mean:.6g}</td>"
            f"<td>{gp_uncorr[i]:.6g}</td>"
            f"<td>{ratio_uncorr:.6g}</td>"
            "</tr>"
        )
    return """
    <table>
      <thead>
        <tr>
          <th>data point</th>
          <th>centrality</th>
          <th>bin</th>
          <th>x_T</th>
          <th>exp mean U(x_T)</th>
          <th>exp stat uncertainty</th>
          <th>GP mean</th>
          <th>ratio GP mean / exp mean</th>
          <th>GP uncorrelated uncertainty</th>
          <th>ratio GP uncorr / exp stat</th>
        </tr>
      </thead>
      <tbody>{rows}</tbody>
    </table>
    """.format(rows="\n".join(rows))


display(HTML(hyperparameter_table_html()))
display(HTML(point_table_html()))

centrality,n points,ell,GP sigma_f log(y),GP sigma_f^2 log(y),GP sigma_n log(y),GP sigma_n^2 log(y),mean exp stat / y,mean (exp stat / y)^2,mean GP uncorr / GP mean,NLL
0-5%,41,0.2,1.25485,1.57466,0.00770553,5.93751e-05,0.11101,0.0126099,0.00770553,12.0971


data point,centrality,bin,x_T,exp mean U(x_T),exp stat uncertainty,GP mean,ratio GP mean / exp mean,GP uncorrelated uncertainty,ratio GP uncorr / exp stat
0,0-5%,0,0.212766,0.752186,0.119005,0.752762,1.00077,0.00580043,0.0487409
1,0-5%,1,0.251451,0.808867,0.121065,0.805458,0.995787,0.00620648,0.0512658
2,0-5%,2,0.290135,0.836254,0.124831,0.839625,1.00403,0.00646975,0.0518279
3,0-5%,3,0.32882,0.847555,0.126331,0.853577,1.0071,0.00657726,0.0520637
4,0-5%,4,0.367505,0.86054,0.128202,0.849221,0.986847,0.00654369,0.051042
5,0-5%,5,0.435203,0.80757,0.0887911,0.81378,1.00769,0.0062706,0.070622
6,0-5%,6,0.531915,0.753526,0.0836087,0.751355,0.997119,0.00578959,0.0692463
7,0-5%,7,0.628627,0.694075,0.0781993,0.694401,1.00047,0.00535072,0.0684241
8,0-5%,8,0.725338,0.630514,0.0687159,0.631075,1.00089,0.00486277,0.0707662
9,0-5%,9,0.82205,0.567674,0.0608835,0.566886,0.998613,0.00436816,0.0717462


In [7]:
import ROOT


ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)


def _draw_one_line(xmin, xmax):
    line = ROOT.TLine(xmin, 1.0, xmax, 1.0)
    line.SetLineStyle(2)
    line.SetLineColor(ROOT.kGray + 2)
    line.SetLineWidth(1)
    line.Draw("SAME")
    return line


def plot_gp_vs_exp_mean_root_inline(fit):
    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"c_alice_grad_gp_mean_{uid}", "ALICE Grad: GP mean vs experimental central values", 1000, 650)
    c.SetLogy()
    c.SetGridx(True)
    c.SetGridy(True)

    x = fit["x"].astype(np.float64)
    y = fit["y"].astype(np.float64)
    exp_unc = fit["exp_uncorr"].astype(np.float64)
    x_grid = fit["x_grid"].astype(np.float64)
    gp_mean_grid = fit["gp_mean_grid"].astype(np.float64)

    ymin = max(float(np.min(y - exp_unc)), float(np.min(y[y > 0]) * 1e-3))
    ymax = float(max(np.max(y + exp_unc), np.max(gp_mean_grid)) * 1.6)
    frame = c.DrawFrame(float(x.min()), ymin, float(x.max()), ymax)
    frame.SetTitle(f"ALICE Pb-Pb 2.76 TeV Grad, {SELECTED_CENTRALITY};x_{{T}};U(x_{{T}})")

    zeros = np.zeros_like(x)
    g_exp = ROOT.TGraphAsymmErrors(len(x), x, y, zeros, zeros, exp_unc, exp_unc)
    g_exp.SetName(f"g_exp_mean_{uid}")
    g_exp.SetMarkerStyle(20)
    g_exp.SetMarkerSize(0.9)
    g_exp.SetMarkerColor(ROOT.kBlack)
    g_exp.SetLineColor(ROOT.kOrange + 7)
    g_exp.SetLineWidth(2)

    g_gp = ROOT.TGraph(len(x_grid), x_grid, gp_mean_grid)
    g_gp.SetName(f"g_gp_mean_{uid}")
    g_gp.SetLineColor(ROOT.kAzure + 2)
    g_gp.SetLineWidth(3)

    leg = ROOT.TLegend(0.52, 0.72, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.AddEntry(g_exp, "exp mean + stat uncertainty", "lep")
    leg.AddEntry(g_gp, "GP mean", "l")

    g_gp.Draw("L SAME")
    g_exp.Draw("P SAME")
    leg.Draw()

    c._keep = [frame, g_exp, g_gp, leg]
    c.Modified()
    c.Update()
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


def plot_gp_uncorr_vs_exp_uncorr_root_inline(fit):
    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"c_alice_grad_uncorr_{uid}", "ALICE Grad: GP uncorrelated uncertainty vs experimental statistical uncertainty", 1000, 650)
    c.SetLogy()
    c.SetGridx(True)
    c.SetGridy(True)

    x = fit["x"].astype(np.float64)
    exp_unc = fit["exp_uncorr"].astype(np.float64)
    gp_unc = fit["gp_uncorr_std_train"].astype(np.float64)

    ymin = max(float(min(exp_unc.min(), gp_unc.min()) * 0.5), 1e-12)
    ymax = float(max(exp_unc.max(), gp_unc.max()) * 2.0)
    frame = c.DrawFrame(float(x.min()), ymin, float(x.max()), ymax)
    frame.SetTitle(f"ALICE Pb-Pb 2.76 TeV Grad, {SELECTED_CENTRALITY};x_{{T}};uncorrelated uncertainty")

    g_exp = ROOT.TGraph(len(x), x, exp_unc)
    g_exp.SetName(f"g_exp_unc_{uid}")
    g_exp.SetMarkerStyle(20)
    g_exp.SetMarkerSize(0.85)
    g_exp.SetMarkerColor(ROOT.kOrange + 7)
    g_exp.SetLineColor(ROOT.kOrange + 7)
    g_exp.SetLineWidth(2)

    g_gp = ROOT.TGraph(len(x), x, gp_unc)
    g_gp.SetName(f"g_gp_unc_{uid}")
    g_gp.SetMarkerStyle(21)
    g_gp.SetMarkerSize(0.80)
    g_gp.SetMarkerColor(ROOT.kAzure + 2)
    g_gp.SetLineColor(ROOT.kAzure + 2)
    g_gp.SetLineWidth(2)

    leg = ROOT.TLegend(0.52, 0.72, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.AddEntry(g_exp, "experimental stat uncertainty", "lp")
    leg.AddEntry(g_gp, "GP white-noise uncertainty", "lp")

    g_exp.Draw("LP SAME")
    g_gp.Draw("LP SAME")
    leg.Draw()

    c._keep = [frame, g_exp, g_gp, leg]
    c.Modified()
    c.Update()
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


def plot_ratio_summary_root_inline(fit):
    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"c_alice_grad_ratios_{uid}", "ALICE Grad: GP/experiment ratios", 1000, 650)
    c.SetGridx(True)
    c.SetGridy(True)

    x = fit["x"].astype(np.float64)
    ratio_mean = safe_ratio(fit["gp_mean_train"], fit["y"]).astype(np.float64)
    ratio_uncorr = safe_ratio(fit["gp_uncorr_std_train"], fit["exp_uncorr"]).astype(np.float64)
    finite = np.concatenate([ratio_mean[np.isfinite(ratio_mean)], ratio_uncorr[np.isfinite(ratio_uncorr)]])
    max_dev = float(np.max(np.abs(finite - 1.0)))
    ymin = 1.0 - 1.2 * max_dev
    ymax = 1.0 + 1.2 * max_dev
    if ymin == ymax:
        ymin, ymax = 0.9, 1.1

    frame = c.DrawFrame(float(x.min()), ymin, float(x.max()), ymax)
    frame.SetTitle(f"ALICE Pb-Pb 2.76 TeV Grad, {SELECTED_CENTRALITY};x_{{T}};ratio")
    one = _draw_one_line(float(x.min()), float(x.max()))

    g_mean = ROOT.TGraph(len(x), x, ratio_mean)
    g_mean.SetName(f"g_ratio_mean_{uid}")
    g_mean.SetMarkerStyle(20)
    g_mean.SetMarkerSize(0.75)
    g_mean.SetMarkerColor(ROOT.kBlack)
    g_mean.SetLineColor(ROOT.kBlack)
    g_mean.SetLineWidth(2)

    g_unc = ROOT.TGraph(len(x), x, ratio_uncorr)
    g_unc.SetName(f"g_ratio_unc_{uid}")
    g_unc.SetMarkerStyle(21)
    g_unc.SetMarkerSize(0.75)
    g_unc.SetMarkerColor(ROOT.kAzure + 2)
    g_unc.SetLineColor(ROOT.kAzure + 2)
    g_unc.SetLineWidth(2)

    leg = ROOT.TLegend(0.52, 0.72, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.AddEntry(g_mean, "GP mean / exp mean", "lp")
    leg.AddEntry(g_unc, "GP uncorr / exp stat", "lp")

    g_mean.Draw("LP SAME")
    g_unc.Draw("LP SAME")
    leg.Draw()

    c._keep = [frame, one, g_mean, g_unc, leg]
    c.Modified()
    c.Update()
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_alice_grad_gp_mean = plot_gp_vs_exp_mean_root_inline(fit)
display(c_alice_grad_gp_mean)

c_alice_grad_uncorr = plot_gp_uncorr_vs_exp_uncorr_root_inline(fit)
display(c_alice_grad_uncorr)

c_alice_grad_ratios = plot_ratio_summary_root_inline(fit)
display(c_alice_grad_ratios)

## GP fit with optimized correlation length

This section repeats the ALICE Grad fit with `sigma_f`, `ell`, and `sigma_n` optimized together, then plots the three ratios for the optimized length scale.


In [9]:
# GP with optimized RBF correlation length for the selected ALICE Grad centrality.
def negative_log_marginal_likelihood_free_ell(theta):
    log_sigma_f, log_ell, log_sigma_n = theta
    sigma_f_try = np.exp(log_sigma_f)
    ell_try = np.exp(log_ell)
    sigma_n_try = np.exp(log_sigma_n)

    log_y = np.log(y)
    z = log_y - np.mean(log_y)
    R_try = rbf_correlation(x, x, length_scale=ell_try)
    K_try = sigma_f_try**2 * R_try + sigma_n_try**2 * np.eye(len(x))
    L_try = stable_cholesky(K_try)
    alpha_try = np.linalg.solve(L_try.T, np.linalg.solve(L_try, z))
    log_det = 2.0 * np.sum(np.log(np.diag(L_try)))
    return 0.5 * z @ alpha_try + 0.5 * log_det + 0.5 * len(x) * np.log(2.0 * np.pi)


def fit_gp_free_ell(x, y):
    log_y = np.log(y)
    z = log_y - np.mean(log_y)
    z_std = max(float(np.std(z)), 1e-8)
    initial = np.log([fit["sigma_f"], LENGTH_SCALE, fit["sigma_n"]])
    bounds = [
        (np.log(1e-4 * z_std), np.log(20.0 * z_std)),
        (np.log(0.02), np.log(10.0)),
        (np.log(1e-6 * z_std), np.log(5.0 * z_std)),
    ]

    if minimize is not None:
        starts = [
            initial,
            np.log([z_std, 0.05, 0.01 * z_std]),
            np.log([z_std, 0.2, 0.05 * z_std]),
            np.log([2.0 * z_std, 1.0, 0.10 * z_std]),
        ]
        results = []
        for start in starts:
            result = minimize(
                negative_log_marginal_likelihood_free_ell,
                start,
                method="L-BFGS-B",
                bounds=bounds,
                options={"maxiter": 10000, "ftol": 1e-12, "gtol": 1e-8},
            )
            if result.success and np.isfinite(result.fun):
                results.append(result)
        if results:
            best = min(results, key=lambda item: item.fun)
            sigma_f_best, ell_best, sigma_n_best = np.exp(best.x)
            method = "scipy L-BFGS-B, free ell"
            nll_best = float(best.fun)
        else:
            sigma_f_best = ell_best = sigma_n_best = nll_best = None
            method = None
    else:
        sigma_f_best = ell_best = sigma_n_best = nll_best = None
        method = None

    if method is None:
        ell_grid = np.round(np.arange(0.05, 2.0 + 0.025, 0.05), 2)
        best_fit = None
        for ell_try in ell_grid:
            candidate = fit_gp_fixed_ell(x, y, length_scale=float(ell_try))
            if best_fit is None or candidate["nll"] < best_fit["nll"]:
                best_fit = candidate
                ell_best = float(ell_try)
        sigma_f_best = best_fit["sigma_f"]
        sigma_n_best = best_fit["sigma_n"]
        nll_best = best_fit["nll"]
        method = "profile scan fallback over ell = 0.05 ... 2.0"

    return fit_gp_fixed_ell(x, y, length_scale=float(ell_best)) | {
        "length_scale": float(ell_best),
        "fit_method_free_ell": method,
        "nll_free_ell": float(nll_best),
    }


fit_free_ell = fit_gp_free_ell(x, y)
fit_free_ell["exp_uncorr"] = exp_uncorr
fit_free_ell["records"] = group

ratio_mean_free_ell = safe_ratio(fit_free_ell["gp_mean_train"], y)
ratio_uncorr_free_ell = safe_ratio(fit_free_ell["gp_uncorr_std_train"], exp_uncorr)
print(f"Free-ell fit method: {fit_free_ell['fit_method_free_ell']}")
print(f"optimized ell: {fit_free_ell['length_scale']:.6g}")
print(f"sigma_f={fit_free_ell['sigma_f']:.6g}, sigma_f^2={fit_free_ell['sigma_f2']:.6g}")
print(f"sigma_n={fit_free_ell['sigma_n']:.6g}, sigma_n^2={fit_free_ell['sigma_n2']:.6g}")
print(f"NLL={fit_free_ell['nll_free_ell']:.6g}")


def free_ell_summary_table_html():
    return f"""
    <table>
      <thead>
        <tr>
          <th>centrality</th>
          <th>optimized ell</th>
          <th>sigma_f log(y)</th>
          <th>sigma_f^2 log(y)</th>
          <th>sigma_n log(y)</th>
          <th>sigma_n^2 log(y)</th>
          <th>mean GP mean / exp mean</th>
          <th>mean GP uncorr / exp stat</th>
          <th>NLL</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td>{SELECTED_CENTRALITY}</td>
          <td>{fit_free_ell['length_scale']:.6g}</td>
          <td>{fit_free_ell['sigma_f']:.6g}</td>
          <td>{fit_free_ell['sigma_f2']:.6g}</td>
          <td>{fit_free_ell['sigma_n']:.6g}</td>
          <td>{fit_free_ell['sigma_n2']:.6g}</td>
          <td>{np.nanmean(ratio_mean_free_ell):.6g}</td>
          <td>{np.nanmean(ratio_uncorr_free_ell):.6g}</td>
          <td>{fit_free_ell['nll_free_ell']:.6g}</td>
        </tr>
      </tbody>
    </table>
    """


display(HTML(free_ell_summary_table_html()))


def plot_free_ell_two_ratios_root_inline():
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"c_alice_grad_free_ell_two_ratios_{uid}", "ALICE Grad free-ell ratios", 1050, 650)
    c.SetGridx(True)
    c.SetGridy(True)

    x_plot = x.astype(np.float64)
    curves = [
        ("GP mean / exp mean", ratio_mean_free_ell.astype(np.float64), ROOT.kBlack, 20),
        ("GP uncorr / exp stat", ratio_uncorr_free_ell.astype(np.float64), ROOT.kAzure + 2, 21),
    ]
    finite = np.concatenate([values[np.isfinite(values)] for _, values, _, _ in curves])
    max_dev = float(np.max(np.abs(finite - 1.0)))
    ymin = 1.0 - 1.20 * max_dev
    ymax = 1.0 + 1.20 * max_dev
    if ymin == ymax:
        ymin, ymax = 0.9, 1.1

    frame = c.DrawFrame(float(x_plot.min()), ymin, float(x_plot.max()), ymax)
    frame.SetTitle(f"ALICE Pb-Pb 2.76 TeV Grad, {SELECTED_CENTRALITY}, optimized ell = {fit_free_ell['length_scale']:.4g};x_{{T}};ratio")

    one = ROOT.TLine(float(x_plot.min()), 1.0, float(x_plot.max()), 1.0)
    one.SetLineStyle(2)
    one.SetLineColor(ROOT.kGray + 2)
    one.SetLineWidth(1)
    one.Draw("SAME")

    leg = ROOT.TLegend(0.52, 0.68, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    keep = [frame, one, leg]

    for i, (label, values, color, marker) in enumerate(curves):
        g = ROOT.TGraph(len(x_plot), x_plot, values)
        g.SetName(f"g_alice_free_ell_ratio_{i}_{uid}")
        g.SetMarkerStyle(marker)
        g.SetMarkerSize(0.75)
        g.SetMarkerColor(color)
        g.SetLineColor(color)
        g.SetLineWidth(2)
        g.Draw("LP SAME")
        leg.AddEntry(g, label, "lp")
        keep.append(g)

    leg.Draw()
    c._keep = keep
    c.Modified()
    c.Update()
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_alice_grad_free_ell_two_ratios = plot_free_ell_two_ratios_root_inline()
display(c_alice_grad_free_ell_two_ratios)


Free-ell fit method: scipy L-BFGS-B, free ell
optimized ell: 1.07088
sigma_f=2.52185, sigma_f^2=6.35972
sigma_n=0.0164112, sigma_n^2=0.000269327
NLL=-64.7196


centrality,optimized ell,sigma_f log(y),sigma_f^2 log(y),sigma_n log(y),sigma_n^2 log(y),mean GP mean / exp mean,mean GP uncorr / exp stat,NLL
0-5%,1.07088,2.52185,6.35972,0.0164112,0.000269327,1.0001,0.150772,-64.7196
